# LAB-06(SPII)

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from scipy.stats import chi2

In [2]:
# Load dataset
from statsmodels.datasets import get_rdataset

In [3]:
# Load the "mtcars" dataset from R
mtcars = get_rdataset("mtcars", "datasets").data

In [4]:
mtcars

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
rownames,,,,,,,,,,,
Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2
Valiant,18.1,6,225.0,105,2.76,3.460,20.22,1,0,3,1
Duster 360,14.3,8,360.0,245,3.21,3.570,15.84,0,0,3,4
Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2
Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2


In [5]:
mtcars.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
rownames,,,,,,,,,,,
Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [6]:
mtcars.isnull().sum()

mpg     0
cyl     0
disp    0
hp      0
drat    0
wt      0
qsec    0
vs      0
am      0
gear    0
carb    0
dtype: int64

In [7]:
mtcars.dtypes

mpg     float64
cyl       int64
disp    float64
hp        int64
drat    float64
wt      float64
qsec    float64
vs        int64
am        int64
gear      int64
carb      int64
dtype: object

## 06(a)

In [8]:
# Create observational vector
mtcars['state'] = np.where(mtcars['mpg'] >= 19, 1, 0)

In [9]:
mtcars

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb,state
rownames,,,,,,,,,,,,
Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4,1
Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4,1
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1,1
Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1,1
Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2,0
Valiant,18.1,6,225.0,105,2.76,3.460,20.22,1,0,3,1,0
Duster 360,14.3,8,360.0,245,3.21,3.570,15.84,0,0,3,4,0
Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2,1
Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2,1


## 06(b)

In [10]:
# Compute transition counts
states = mtcars['state'].values
transitions = np.zeros((2, 2))  # 2x2 matrix for states {0,1}
for i in range(len(states) - 1):
    transitions[states[i], states[i+1]] += 1
print("Transitions:")
print(transitions)

Transitions:
[[10.  5.]
 [ 5. 11.]]


In [11]:
# Compute transition probability matrix
transition_matrix = transitions / transitions.sum(axis=1, keepdims=True)
print("Estimated Transition Probability Matrix:")
print(transition_matrix)

Estimated Transition Probability Matrix:
[[0.66666667 0.33333333]
 [0.3125     0.6875    ]]


## 06(c)

In [12]:
# Compare with given matrix
expected_matrix = np.array([[0.2, 0.8], [0.5, 0.5]])
print("Expected_matrix:")
print(expected_matrix)

Expected_matrix:
[[0.2 0.8]
 [0.5 0.5]]


In [13]:
# Row totals
row_totals = np.sum(transitions, axis=1)
print(row_totals)

[15. 16.]


In [14]:
# Compute the chi-square statistic
chi_square_stat = np.sum(row_totals[:, None] * (transition_matrix - expected_matrix) ** 2 / expected_matrix)
print(chi_square_stat)

22.666666666666664


In [15]:
# Degrees of freedom (df)
df = (transition_matrix.shape[0] - 1) * (transition_matrix.shape[1] - 1)
print(df)

1


In [16]:
# Significance level (alpha)
alpha = 0.05

### Using Critical Value Approach

In [17]:
# Calculate the chi-square critical (tabulated) value for the given alpha and df
chi_square_critical_value = chi2.ppf(1 - alpha, df)
# Print the chi-square tabulated (critical) value
print(f"Chi-Square Critical Value (Tabulated) at alpha = {alpha} and df = {df}: {chi_square_critical_value}")

Chi-Square Critical Value (Tabulated) at alpha = 0.05 and df = 1: 3.841458820694124


In [18]:
# Interpretation based on critical value
if chi_square_stat > chi_square_critical_value:
    print("\nReject the null hypothesis: The observed transition matrix is significantly different from the expected matrix.")
else:
    print("\nFail to reject the null hypothesis: The observed transition matrix is not significantly different from the expected matrix.")


Reject the null hypothesis: The observed transition matrix is significantly different from the expected matrix.


### Using P-Value Approach

In [19]:
# Calculate the p-value
p_value = 1 - chi2.cdf(chi_square_stat, df)
# Print the p-value
print("p-value:", p_value)

p-value: 1.9268160332819306e-06


In [20]:
# Interpretation based on p-value
if p_value <= alpha:
    print("\nReject the null hypothesis: The observed transition matrix is significantly different from the expected matrix.")
else:
    print("\nFail to reject the null hypothesis: The observed transition matrix is not significantly different from the expected matrix.")


Reject the null hypothesis: The observed transition matrix is significantly different from the expected matrix.
